# 🔬 OpenAmadeus Companion — Kurisu Qwen3-TTS 1.7B Remote Worker (Acelerado)

Este notebook executa o modelo **Kurisu Qwen3-TTS 1.7B** (`starrydark/Kurisu_Qwen3_TTS`) acelerado com **CUDA Graphs** (`faster-qwen3-tts`) em GPUs em nuvem (**Google Colab Pro L4 / A100 / T4**), gerando um túnel HTTPS seguro via **Cloudflare Tunnel** com latência em tempo real (< 400ms TTFA).

### Melhorias Técnicas de Tempo Real:
- **Aceleração CUDA Graphs:** Reduz o passo de inferência do talker de ~330ms para ~54ms (RTF > 1.9x).
- **Streaming Contínuo (`audio/L16;rate=24000`):** Primeiro chunk de áudio emitido em ~390ms.
- **Suavização Contínua $C^0$ nas Costuras:** Rampa de 120 amostras nas bordas dos chunks para eliminar micro-estalos sem perda de amplitude.
- **Warm-up Completo:** Pré-aquecimento do Transformer e do decodificador Mimi, garantindo resposta imediata na 1ª fala.

In [ ]:
# Passo 1: Verificar GPU disponível no Colab
!nvidia-smi

In [ ]:
# Passo 2: Instalar dependências fixadas para CUDA Graphs e Qwen3-TTS
!apt-get update -qq && apt-get install -qq -y sox libsox-fmt-all
!pip install -q --upgrade pip
!pip install -q "transformers==5.15.1" faster-qwen3-tts soundfile fastapi uvicorn pydantic pycloudflared httpx

In [ ]:
# Passo 3: Iniciar Servidor Kurisu Qwen3-TTS com CUDA Graphs e Streaming
import os
import io
import time
import torch
import soundfile as sf
import numpy as np
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, StreamingResponse
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare
from faster_qwen3_tts import FasterQwen3TTSModel

print("=" * 65)
print("⏳ Carregando Kurisu Qwen3-TTS 1.7B com faster-qwen3-tts...")
print("=" * 65)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

t_load_start = time.perf_counter()
model = FasterQwen3TTSModel.from_pretrained(
    "starrydark/Kurisu_Qwen3_TTS",
    device_map=device,
    dtype=dtype,
)
print(f"✅ Modelo carregado na GPU em {round(time.perf_counter() - t_load_start, 2)}s")

# Capturar CUDA Graphs e aquecer o pipeline de síntese
print("⚡ Capturando CUDA Graphs (aceleração do decoder e talker)...")
t_graphs = time.perf_counter()
model.setup_cuda_graphs()
model.warmup(prefill_len=100)
print(f"✅ CUDA Graphs capturados em {round(time.perf_counter() - t_graphs, 2)}s")

# Warm-up completo incluindo o decodificador Mimi
print("🔥 Aquecendo decodificador de áudio Mimi com chunk de teste...")
warmup_gen = model.generate_custom_voice_streaming(
    text="Oi.",
    speaker="kurisu",
    language="Portuguese",
    chunk_size=8,
)
for _ in warmup_gen:
    break
print("✅ Pipeline 100% aquecido! TTFA pronto em ~390ms.")

def apply_boundary_continuity(chunk: np.ndarray, prev_sample: Optional[float], ramp_len: int = 120) -> np.ndarray:
    """
    Aplica rampa aditiva suave nas primeiras ramp_len amostras do chunk atual
    baseado no salto de amplitude com o chunk anterior, zerando descontinuidades sem fade-out.
    """
    if prev_sample is None or len(chunk) == 0:
        return chunk
    actual_ramp = min(ramp_len, len(chunk))
    start_jump = float(prev_sample) - float(chunk[0])
    ramp = np.linspace(1.0, 0.0, actual_ramp, dtype=chunk.dtype)
    chunk[:actual_ramp] += start_jump * ramp
    return chunk

app = FastAPI(title="OpenAmadeus Qwen3-TTS Colab Worker")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SynthesizeRequest(BaseModel):
    text: str
    language: Optional[str] = "Portuguese"
    instruct: Optional[str] = None
    temperature: Optional[float] = 0.75
    top_p: Optional[float] = 0.95
    repetition_penalty: Optional[float] = 1.05

@app.get("/")
def root():
    return {"system": "OpenAmadeus Qwen3-TTS Colab Worker", "status": "ready", "backend": "faster_qwen3_tts"}

@app.get("/health")
def health():
    vram = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
    return {
        "status": "ready",
        "device": device,
        "backend": "faster_qwen3_tts",
        "cuda_available": torch.cuda.is_available(),
        "vram_allocated_mb": round(vram, 2),
        "model": "starrydark/Kurisu_Qwen3_TTS",
    }

@app.post("/synthesize_stream")
def synthesize_stream(req: SynthesizeRequest):
    """
    Transfere fluxo PCM16 bruto em tempo real com TTFA < 400ms e suavização contínua de borda.
    """
    text = (req.text or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="Texto vazio.")

    def pcm_generator():
        prev_sample = None
        with torch.inference_mode():
            stream_gen = model.generate_custom_voice_streaming(
                text=text,
                speaker="kurisu",
                language=req.language or "Portuguese",
                instruct=req.instruct or None,
                chunk_size=8,
                temperature=req.temperature if req.temperature is not None else 0.75,
                top_p=req.top_p if req.top_p is not None else 0.95,
                repetition_penalty=req.repetition_penalty if req.repetition_penalty is not None else 1.05,
            )
            for audio_chunk, sr, _ in stream_gen:
                chunk = audio_chunk.copy()
                chunk = apply_boundary_continuity(chunk, prev_sample)
                prev_sample = float(chunk[-1])
                pcm16 = (np.clip(chunk, -1.0, 1.0) * 32767).astype(np.int16).tobytes()
                yield pcm16

    return StreamingResponse(
        pcm_generator(),
        media_type="audio/L16;rate=24000",
        headers={
            "X-Audio-Format": "pcm_s16le",
            "X-Sample-Rate": "24000",
            "X-Backend": "faster_qwen3_tts",
            "X-Accel-Buffering": "no",
        },
    )

@app.post("/synthesize")
def synthesize(req: SynthesizeRequest):
    text = (req.text or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="Texto vazio.")

    t0 = time.perf_counter()
    with torch.inference_mode():
        wavs, sr = model.generate_custom_voice(
            text=text,
            speaker="kurisu",
            language=req.language or "Portuguese",
            instruct=req.instruct or None,
            temperature=req.temperature if req.temperature is not None else 0.75,
            top_p=req.top_p if req.top_p is not None else 0.95,
            repetition_penalty=req.repetition_penalty if req.repetition_penalty is not None else 1.05,
        )

    audio = wavs[0]
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    if audio.ndim > 1:
        audio = audio.squeeze()

    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    buf.seek(0)
    elapsed = round(time.perf_counter() - t0, 3)
    print(f"🎙️ Síntese: {len(text)} caracteres em {elapsed}s")
    return Response(
        content=buf.read(),
        media_type="audio/wav",
        headers={"X-Backend": "faster_qwen3_tts"},
    )

# Abrir Túnel Cloudflare na porta 8005
PORT = 8005
tunnel_res = try_cloudflare(port=PORT)
tunnel_url = tunnel_res.tunnel if hasattr(tunnel_res, 'tunnel') else str(tunnel_res)
print("\n" + "=" * 65)
print("  🌟 URL PÚBLICA DO CLOUDFLARE TUNNEL (CUDA Graphs Real-Time):")
print(f"  👉 {tunnel_url}")
print("=" * 65)
print("  Copie e cole este link em 'Configurações > Voz > URL Remota (Colab)' no OpenAmadeus Companion!\n")

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()